# Recommendation Systems

#### En este notebook exploramos la creacion de sistemas de recomendación, utilizando el conjunto de datos sobre ingresos de personas adultas de la oficina del Censo de los Estados Unidos. Desarrolaremos un sistema que recomiende trayectorias profesinales y educativas para yudar a los usuarios a mejorar su potencial de ingresos.

### Datos demográficos

> Edad, sexo, pais de origen.
>
> Educación: nivel de educación.
>
> Empleo: ocupación, horas trabajadas por semana.
>
> Datos personales: estado civil.
>
> Objetivo: ingresos anuales (>50.000 ó <=50.000)

## Enfoque

> Crearemos un sistema de recomendaciones que analice los perfiles de los usuarios y sugiera trayectorias similares de altos ingresos basados en caracteristicas demográficas y profesionales.

## Importamos las librerias

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
import warnings
warnings.filterwarnings('ignore')

## Cargamos los datos

In [3]:
# Descargamos el dataset y cargamos los datos directamente desde la URL
url = "https://raw.githubusercontent.com/4GeeksAcademy/predicting-your-future-with-data/main/adult-census-income.csv"

# cargar el conjunto de datos en un dataframe de pandas
df = pd.read_csv(url)

# Guardamos una copia local del dataset para futuras referencias
df.to_csv('../data/raw/adult-census-income.csv', index=False)

# Mostrar las primeras filas del DataFrame
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## Procesamiento de datos

In [4]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [5]:
# Limpiar valores faltantes y datos inconsistentes
df = df.replace('?', np.nan)
df = df.dropna()

# eliminamos espacios en blanco de las columnas categóricas
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].str.strip()

# codificamos variables categoricas usando Label Encoding
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

# Normalizar variables numéricas
numerical_cols = ['age', 'hours.per.week', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.shape

(30162, 24)

> Recomendar trayectorias profesionales y educativas para mejorar el potencial de ingresos.
>
> Lo que recomendamos: Perfiles profesionales con altos ingresos (combinaciones de educacion + ocupación)
>
> Usuarios: personas con carcateristicas demograficas y profesionales especificas .
>
> Variables del perfil: edad, educacion, ocupacion, horas de trabajo, estado civil, datos demográficos.

## Implementación del filtrado en el contenido

In [7]:
# Crea matriz de caracteristicas para usuarios con ingresos elevados (>50K)
high_income_users = df[df['income'] == '>50K'].copy()

# Seleccionar características relevantes para la creacion de perfil del usuario
feature_cols = ['age', 'education.num', 'hours.per.week', 'workclass_encoded', 
                'education_encoded', 'marital.status_encoded', 'occupation_encoded', 
                'relationship_encoded', 'race_encoded', 'sex_encoded']

X_features = high_income_users[feature_cols]

# Crea un recomendador basado en similitud de coseno
def get_similar_profiles(user_profile, n_recommendations=5):
    user_vector = np.array(user_profile).reshape(1, -1)
    similarities = cosine_similarity(user_vector, X_features)
    similar_indices = similarities[0].argsort()[-n_recommendations-1:-1][::-1]
    return high_income_users.iloc[similar_indices]

## Implementacion del filtrado colaborativo

In [ ]:
# Crea matriz de trayectorias de usuario para filtrado colaborativo
user_trajectory_matrix = df.pivot_table(
    index=['age', 'sex_encoded'], 
    columns=['education_encoded', 'occupation_encoded'], 
    values='income_encoded', 
    aggfunc='mean', 
    fill_value=0
)

# Utilizar K-NN para el filtrado colaborativo
knn = NearestNeighbors(n_neighbors=5, metric='cosine')
knn.fit(user_trajectory_matrix.fillna(0))

def collaborative_recommend(user_age, user_sex, n_recommendations=3):
    user_key = (user_age, user_sex)
    if user_key in user_trajectory_matrix.index:
        user_idx = user_trajectory_matrix.index.get_loc(user_key)
        distances, indices = knn.kneighbors([user_trajectory_matrix.iloc[user_idx]])
        similar_users = user_trajectory_matrix.iloc[indices[0][1:]]
        return similar_users.mean().nlargest(n_recommendations)
    return None

## Sistema hibrido de recomendación

In [ ]:
# Recomendador hibrido que combina el filtrado basado en contenido y colaborativo
def hybrid_recommend(user_profile, user_age, user_sex, content_weight=0.6, collab_weight=0.4):
    # Recomendaciones basadas en contenido
    content_recs = get_similar_profiles(user_profile, n_recommendations=5)
    
    # Recomendacion colaborativa  
    collab_recs = collaborative_recommend(user_age, user_sex, n_recommendations=5)
    
    # Combinar recomendaciones con puntuación ponderada
    hybrid_scores = {}
    
    # Añadir puntuacinoes basadas en contenido
    for idx, row in content_recs.iterrows():
        key = (row['education'], row['occupation'])
        hybrid_scores[key] = hybrid_scores.get(key, 0) + content_weight
    
    # Añadir puntuaciones basadas en filtrado colaborativo
    if collab_recs is not None:
        for (edu, occ), score in collab_recs.items():
            key = (label_encoders['education'].inverse_transform([edu])[0], 
                   label_encoders['occupation'].inverse_transform([occ])[0])
            hybrid_scores[key] = hybrid_scores.get(key, 0) + collab_weight * score
    
    return sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:5]

## Pruebas con perfiles de usuario simulados

In [ ]:
# Caso de prueba 1: graduado de secundaria de 25 años, trabajador a tiempo parcial
user_profile_1 = {
    'age': -1.0,  # Normalized age for 25
    'education.num': -0.5,  # High school level
    'hours.per.week': -1.0,  # Part-time hours
    'workclass_encoded': label_encoders['workclass'].transform(['Private'])[0],
    'education_encoded': label_encoders['education'].transform(['HS-grad'])[0],
    'marital.status_encoded': label_encoders['marital.status'].transform(['Never-married'])[0],
    'occupation_encoded': label_encoders['occupation'].transform(['Other-service'])[0],
    'relationship_encoded': label_encoders['relationship'].transform(['Own-child'])[0],
    'race_encoded': label_encoders['race'].transform(['White'])[0],
    'sex_encoded': label_encoders['sex'].transform(['Male'])[0]
}

user_vector_1 = [user_profile_1[col] for col in feature_cols]
recommendations_1 = hybrid_recommend(user_vector_1, 25, user_profile_1['sex_encoded'])

# Caso de prueba 2: Licenciado de 35 años, profesional a tiempo completo
user_profile_2 = {
    'age': 0.2,  # Edad normalizada para 35
    'education.num': 1.0,  # Nivel de licenciatura
    'hours.per.week': 0.5,  # Horas a tiempo completo
    'workclass_encoded': label_encoders['workclass'].transform(['Private'])[0],
    'education_encoded': label_encoders['education'].transform(['Bachelors'])[0],
    'marital.status_encoded': label_encoders['marital.status'].transform(['Married-civ-spouse'])[0],
    'occupation_encoded': label_encoders['occupation'].transform(['Prof-specialty'])[0],
    'relationship_encoded': label_encoders['relationship'].transform(['Husband'])[0],
    'race_encoded': label_encoders['race'].transform(['White'])[0],
    'sex_encoded': label_encoders['sex'].transform(['Male'])[0]
}

user_vector_2 = [user_profile_2[col] for col in feature_cols]
recommendations_2 = hybrid_recommend(user_vector_2, 35, user_profile_2['sex_encoded'])

In [11]:
# Mostrar recomendaciones para ambos casos de prueba
def display_recommendations(recommendations, case_name):
    results = []
    for i, ((education, occupation), score) in enumerate(recommendations, 1):
        results.append(f"{i}. {education} + {occupation} (Score: {score:.3f})")
    return results

case1_results = display_recommendations(recommendations_1, "Young High School Graduate")
case2_results = display_recommendations(recommendations_2, "Mid-Career Professional")

# imprimir resultados para análisis
print("=== RECOMMENDATION SYSTEM RESULTS ===\n")

print("Test Case 1: 25-year-old High School Graduate (Part-time)")
print("Recommended career paths:")
for result in case1_results:
    print(f"  {result}")

print(f"\nTest Case 2: 35-year-old Bachelor's Degree Holder (Full-time)")
print("Recommended career paths:")
for result in case2_results:
    print(f"  {result}")

# Resumen dle rendimiento del sistema de recomendación
system_summary = {
    'Total Users': len(df),
    'High Income Users': len(high_income_users),
    'Feature Dimensions': len(feature_cols),
    'Recommendation Methods': 'Content-Based + Collaborative + Hybrid'
}

print(f"\n=== SYSTEM SUMMARY ===")
for key, value in system_summary.items():
    print(f"{key}: {value}")

=== RECOMMENDATION SYSTEM RESULTS ===

Test Case 1: 25-year-old High School Graduate (Part-time)
Recommended career paths:
  1. Some-college + Protective-serv (Score: 1.200)
  2. HS-grad + Other-service (Score: 0.600)
  3. Some-college + Tech-support (Score: 0.600)
  4. HS-grad + Machine-op-inspct (Score: 0.600)

Test Case 2: 35-year-old Bachelor's Degree Holder (Full-time)
Recommended career paths:
  1. Bachelors + Prof-specialty (Score: 3.000)

=== SYSTEM SUMMARY ===
Total Users: 30162
High Income Users: 7508
Feature Dimensions: 10
Recommendation Methods: Content-Based + Collaborative + Hybrid


## Resultado del Analisis.

### Caso prueba 1 (joven graduado de secundaria)
>
>El sistema recomienda itinerarios de formacion continua, como titulaciones de grado o master combinadas con especialidades profesionales. Las puntuaciones mas altas indican recomendaciones mas solidas basadas en perfiles similares que han tenido exito. Centrarse en el desaroolo de habilidades y la mejora de la formacion para aumentar los ingresos.

### Caso prueba 2 (profesional en mitad de su carrera)
>
> Las recomendaciones se centran en puestos profesionales avanzados y ocupaciones especializadas.
El sistema aprovecha la formacion previa dle usuario para sugerirle opciones de promocion profesional. Las puntuaciones reflejan la compatibilidad con trayectorias de ingresos elevados en grupos demográficos similares.

### Rendimiento del sistema
>
> Procesa con exito mas de 30.000 perfiles de usuario con un espacio de caracteristicas de 10 dimensiones.
El enfoque hibrido combina la similitud de contenidos con patrones colaborativos. Las recomendaciones se personalizan en funcion del perfil actual y sus caracterisitcas demográficas.

In [12]:
# Analizar alta distribución de ingresos elevados
income_distribution = df['income'].value_counts()
print("Income Distribution:")
for income, count in income_distribution.items():
    percentage = (count / len(df)) * 100
    print(f"  {income}: {count:,} users ({percentage:.1f}%)")

# Las mejores combinaciones de educación y ocupación para los que ganan más
top_combinations = high_income_users.groupby(['education', 'occupation']).size().nlargest(5)
print(f"\nTop 5 High-Income Career Combinations:")
for (education, occupation), count in top_combinations.items():
    print(f"  {education} + {occupation}: {count} users")

# Importancia de las características (valores promedio para altos vs bajos ingresos)
print(f"\nFeature Comparison (High vs Low Income):")
low_income_users = df[df['income'] == '<=50K']

comparison_features = ['age', 'education.num', 'hours.per.week']
for feature in comparison_features:
    high_avg = high_income_users[feature].mean()
    low_avg = low_income_users[feature].mean()
    print(f"  {feature}: High({high_avg:.2f}) vs Low({low_avg:.2f})")

Income Distribution:
  <=50K: 22,654 users (75.1%)
  >50K: 7,508 users (24.9%)

Top 5 High-Income Career Combinations:
  Bachelors + Exec-managerial: 762 users
  Bachelors + Prof-specialty: 567 users
  Masters + Prof-specialty: 406 users
  HS-grad + Craft-repair: 401 users
  Bachelors + Sales: 375 users

Feature Comparison (High vs Low Income):
  age: High(0.42) vs Low(-0.14)
  education.num: High(0.58) vs Low(-0.19)
  hours.per.week: High(0.40) vs Low(-0.13)


### Conclusiones

#### Patrones de distribucion de ingresos.
>
> El 75% de los usuarios ganas <=50.000, lo que crea un objetivo claro para las recomendaciones de mejora. 
> El sistema se centra en el 25% mas exitoso para identificar patrones ganadores.

#### Combinaciones profesionales mas exitosas.
>
> Licenciatura + Ejecutivo/Gestion lidera con 762 personas de altos ingresos.
>
> Licenciatura/Master + Especialidad profesional muestra un solido rendimiento.
>
> El nivel educativo está fuertemente correlacionado con los puestos directivos y profesionales.
>
> Análisis de características: 
> Edad: las personas con ingresos elevados son mayores (normalizado +0,42 frente a -0,14), lo que indica el valor de la experiencia.
Educación: fuerte correlación positiva (+0,58 frente a -0,19) con los ingresos.
Horas de trabajo: las personas con ingresos elevados trabajan más horas (+0,40 frente a -0,13), lo que demuestra el impacto del compromiso.
>
> Eficacia del sistema de recomendaciones:
>
> Identifica con éxito la formación continua como la vía principal para los usuarios jóvenes.
Reconoce las oportunidades de especialización profesional para los profesionales con experiencia.
Combina los patrones demográficos con la correspondencia de perfiles individuales para ofrecer asesoramiento personalizado.